<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Answer

**Unit of analysis:** One row represents one content page (`content_hash_id`) for one client (`client_hash_id`) on one reporting date.

**Time window:** For this assignment I use a mid-panel month (`2026-03`) for exploration and feature engineering. This avoids using the final month as the development window, following the assignment guidance.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
import os
import subprocess
import duckdb
from google.colab import userdata

# Clone the starter repo if needed
REPO = "flyrank-ml-internship-starter"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/flyrank-bih/flyrank-ml-internship-starter"],
        check=True,
    )

os.chdir(REPO)

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {fact_daily}
""").df()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content,start_date,end_date
0,9841378,331437,2026-03-01,2026-03-31


## Answer

### Feature
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- content_visible_query_count
- rare_impressions_share

These are available before making the prediction and can be used as model features.

### Label / Proxy
- is_declining

This is a proxy label that indicates whether impressions in the last 30 days dropped below 80% of the previous 30 days.

### Context
- client_hash_id
- content_hash_id
- report_date

These fields identify, group, or join the data but should not be used directly as model features.

### Excluded
- Future performance values
- Any label-derived information

These are excluded because they would leak future information into the model and produce misleadingly good results.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {fact_daily}
""").df()

,rows,clients,content_items
0,9841378,55,331437


## Answer

The following queries verify the data contract:

1. **Grain:** Check that each client, content page, and reporting date combination appears only once.
2. **Counts and window:** Confirm the total number of rows and the reporting date range.
3. **Missing values:** Check whether important fields contain missing values.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# 1. Verify the grain (should return zero rows)
print("=== Grain Check ===")
grain = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicates
FROM {fact_daily}
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print(grain)

# 2. Verify counts and date window
print("\n=== Counts & Date Window ===")
counts = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {fact_daily}
""").df()

print(counts)

# 3. Check missing values
print("\n=== Missing Values ===")
missing = con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS missing_impressions,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_clicks,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS missing_position
FROM {fact_daily}
""").df()

print(missing)

=== Grain Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, duplicates]
Index: []

=== Counts & Date Window ===
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31

=== Missing Values ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   missing_impressions  missing_clicks  missing_position
0                  0.0             0.0          0.633074


## Answer

### Data limits

- Different clients have different amounts of historical data, so comparisons across clients may not always be fair.
- Some clients only have Google Search Console (GSC) data, while others have both GSC and GA4 data.
- This analysis uses only one month (March 2026), so seasonal trends and long-term behavior are not captured.
- The data is observational and supports decision-making, but it cannot prove cause-and-effect relationships.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# One simple check supporting the limitation about client history

con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS total_clients,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM {fact_daily}
""").df()

,total_clients,earliest_date,latest_date
0,55,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.